In [1]:
# Cel tego notebooka: sprawdzić hipotezę "inny dtype/units NMDB" jako
# wyjaśnienie niezgodności naszej replikacji artykułu dla Moskwy przy
# PODANYM w artykule t0 (bez skanowania - patrz rozmowa 2026-08-19,
# wracamy do sprawdzenia punktowego sprzed 20260717.txt pkt 7, NIE do
# skanu po t0 z 20260819a.ipynb - to osobny wątek na później).
#
# Przypomnienie ustalonego przepisu (sekcja 3 artykułu - NIE sekcja 4/
# Fig.4, to była pomyłka z 20260819a.ipynb, patrz notatka w pamięci
# projektu): P=1675 dni, d=5 dni, m=4.0, Δt(lag)=15 dni,
# t0=14 lis 2013 07:00:00 GMT. Artykuł: N+=218, N-=113, PCDF=4.1e-9
# (~6 sigma). U nas dotychczas (mosc_data.csv, dtype=corr_for_efficiency,
# units=0): N+=206, N-=128, PCDF=1.16e-5 (sigma=4.23) - patrz
# 20260717.txt pkt 7.
import numpy as np
import pandas as pd
from scipy.stats import binom, norm

USGS_PATH = "../data/usgs_data/usgs_m4_2005_2025.csv"

def load_earthquakes(min_mag=4.0):
    df = pd.read_csv(USGS_PATH, usecols=["time", "mag"])
    df["time"] = pd.to_datetime(df["time"], utc=True).dt.tz_localize(None)
    df = df[df["mag"] >= min_mag]
    return df.set_index("time")["mag"].sort_index()


def cosmoseismic_stat(cr, eq, t0, P_days, d_days, m, dt_days):
    N = int(P_days // d_days)
    edges = pd.date_range(t0, periods=N + 1, freq=pd.Timedelta(days=d_days))
    eq_edges = edges + pd.Timedelta(days=dt_days)

    cr_cats = pd.cut(cr.index, edges, right=False)
    cr_binned = cr.groupby(cr_cats, observed=False).mean().reindex(cr_cats.categories)
    cr_vals = cr_binned.to_numpy()

    eq_in_range = eq[(eq.index >= eq_edges[0]) & (eq.index < eq_edges[-1])]
    eq_cats = pd.cut(eq_in_range.index, eq_edges, right=False)
    eq_binned = eq_in_range.groupby(eq_cats, observed=False).sum().reindex(eq_cats.categories, fill_value=0.0)
    sm_vals = eq_binned.to_numpy()

    nCR_i, nCR_im1 = cr_vals[1:], cr_vals[:-1]
    dCR = nCR_i - nCR_im1
    Sm = sm_vals[1:]

    med_Sm = np.nanmedian(Sm)
    med_dCR = np.nanmedian(np.abs(dCR))

    A = Sm / med_Sm - 1
    B = np.abs(dCR) / med_dCR - 1

    valid = (
        (A != 0) & (B != 0) &
        (nCR_i > 0) & (nCR_im1 > 0) &
        (Sm > 0) &
        ~np.isnan(A) & ~np.isnan(B)
    )

    c_valid = (A * B)[valid]
    Np, Nm = int((c_valid > 0).sum()), int((c_valid < 0).sum())
    n_total = Np + Nm

    if n_total == 0:
        return dict(N=N, N_valid=0, Np=0, Nm=0, PPDF=np.nan, PCDF=np.nan, sigma=np.nan)

    ppdf = binom.pmf(Np, n_total, 0.5)
    pcdf = binom.sf(Np - 1, n_total, 0.5)
    sigma = norm.isf(pcdf)

    return dict(N=N, N_valid=n_total, Np=Np, Nm=Nm, PPDF=ppdf, PCDF=pcdf, sigma=sigma)


eq = load_earthquakes(min_mag=4.0)
print(f"EQ (M>=4.0): {len(eq)} zdarzeń, {eq.index.min()} .. {eq.index.max()}")

T0 = pd.Timestamp("2013-11-14 07:00:00")
P_DAYS = 1675
D_DAYS = 5
DT_DAYS = 15
M_THRESHOLD = 4.0

ARTICLE_RESULT = dict(Np=218, Nm=113, PCDF=4.1e-9)
BASELINE_RESULT = dict(Np=206, Nm=128, PCDF=1.16e-5)  # mosc_data.csv, corr_for_efficiency/units=0, z 20260717.txt


EQ (M>=4.0): 290945 zdarzeń, 2005-01-01 00:47:34.620000 .. 2025-01-31 23:57:39.481000


In [2]:
# Wczytanie 4 wariantów dtype/units (pobranych przez
# skrypty_dl/wlasne/download_mosc_dtype_check.py, tylko dla okna
# 2013-2018 potrzebnego do tego testu) + dotychczasowego mosc_data.csv
# (pełna historia) jako niezależnej kontroli.
VARIANTS = {
    "eff_0 (dotychczasowy std., kontrola)": "../data/mosc_dtype_check/mosc_eff_0.csv",
    "eff_1": "../data/mosc_dtype_check/mosc_eff_1.csv",
    "press_0": "../data/mosc_dtype_check/mosc_press_0.csv",
    "press_1": "../data/mosc_dtype_check/mosc_press_1.csv",
}

def load_variant(path):
    df = pd.read_csv(path)
    df["datetime"] = pd.to_datetime(df["datetime"])
    return df.set_index("datetime").sort_index()["value"]

cr_variants = {}
for name, path in VARIANTS.items():
    s = load_variant(path)
    n_nonpositive = int((s <= 0).sum())
    print(f"{name}: {len(s)} pomiarów, {s.index.min()} .. {s.index.max()}, "
          f"zakres wartości [{s.min():.4g}, {s.max():.4g}], "
          f"{n_nonpositive} wartości <=0 "
          f"({'UWAGA: test wymaga nCR>0, to odfiltruje część binów' if n_nonpositive else 'OK'})")
    cr_variants[name] = s

# Niezależna kontrola: nasz dotychczasowy pełny mosc_data.csv (ten sam
# dtype/units co eff_0, ale inne pobranie/zakres) - powinien dać dokładnie
# baseline (206/128) z 20260717.txt.
cr_mosc_full = load_variant("../data/mosc_data.csv")
print(f"\nmosc_data.csv (pełna historia, kontrola niezależna): {len(cr_mosc_full)} pomiarów")


eff_0 (dotychczasowy std., kontrola): 8741 pomiarów, 2013-01-01 00:00:00 .. 2018-12-31 18:00:00, zakres wartości [136.2, 246.2], 0 wartości <=0 (OK)
eff_1: 8741 pomiarów, 2013-01-01 00:00:00 .. 2018-12-31 18:00:00, zakres wartości [-41.92, 4.546], 4018 wartości <=0 (UWAGA: test wymaga nCR>0, to odfiltruje część binów)
press_0: 8741 pomiarów, 2013-01-01 00:00:00 .. 2018-12-31 18:00:00, zakres wartości [136.2, 246.2], 0 wartości <=0 (OK)
press_1: 8741 pomiarów, 2013-01-01 00:00:00 .. 2018-12-31 18:00:00, zakres wartości [-41.92, 4.546], 4018 wartości <=0 (UWAGA: test wymaga nCR>0, to odfiltruje część binów)

mosc_data.csv (pełna historia, kontrola niezależna): 91824 pomiarów


In [3]:
# Porównanie: dla każdego wariantu liczymy N+/N-/PCDF przy DOKŁADNIE tym
# samym przepisie co artykuł (P=1675, d=5, m=4.0, dt=15, t0=14 lis 2013),
# i zestawiamy z artykułem oraz z dotychczasowym baseline.
rows = []
rows.append(dict(wariant="ARTYKUŁ (referencja)", Np=ARTICLE_RESULT["Np"], Nm=ARTICLE_RESULT["Nm"],
                  PCDF=ARTICLE_RESULT["PCDF"], sigma=norm.isf(ARTICLE_RESULT["PCDF"])))
rows.append(dict(wariant="baseline 20260717 (referencja, z dziennika)", Np=BASELINE_RESULT["Np"],
                  Nm=BASELINE_RESULT["Nm"], PCDF=BASELINE_RESULT["PCDF"],
                  sigma=norm.isf(BASELINE_RESULT["PCDF"])))

r_full = cosmoseismic_stat(cr_mosc_full, eq, T0, P_DAYS, D_DAYS, M_THRESHOLD, DT_DAYS)
rows.append(dict(wariant="mosc_data.csv (kontrola niezależna, powinno ~= baseline)",
                  Np=r_full["Np"], Nm=r_full["Nm"], PCDF=r_full["PCDF"], sigma=r_full["sigma"]))

for name, cr in cr_variants.items():
    r = cosmoseismic_stat(cr, eq, T0, P_DAYS, D_DAYS, M_THRESHOLD, DT_DAYS)
    rows.append(dict(wariant=name, Np=r["Np"], Nm=r["Nm"], PCDF=r["PCDF"], sigma=r["sigma"]))

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))


                                                 wariant  Np  Nm         PCDF    sigma
                                    ARTYKUŁ (referencja) 218 113 4.100000e-09 5.764295
             baseline 20260717 (referencja, z dziennika) 206 128 1.160000e-05 4.231634
mosc_data.csv (kontrola niezależna, powinno ~= baseline) 206 128 1.162192e-05 4.231209
                    eff_0 (dotychczasowy std., kontrola) 206 128 1.162192e-05 4.231209
                                                   eff_1  81  42 2.780462e-04 3.452172
                                                 press_0 206 128 1.162192e-05 4.231209
                                                 press_1  81  42 2.780462e-04 3.452172
